# Feature Engineering — Churn Bancário

## Objetivo
Transformar os dados brutos em features otimizadas para modelagem,
baseado nos insights obtidos no EDA.

In [6]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import LabelEncoder, StandardScaler

pd.set_option('display.max_columns', None)
print("Bibliotecas carregadas!")

Bibliotecas carregadas!


## 1. Carregando os Dados

In [7]:
df = pd.read_csv('../data/raw/train.csv')
print(f"Shape original: {df.shape}")
df.head()

Shape original: (165034, 14)


,id,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,0,15674932,Okwudilichukwu,668,France,Male,33.0,3,0.00,2,1.0,0.0,181449.97,0
1,1,15749177,Okwudiliolisa,627,France,Male,33.0,1,0.00,2,1.0,1.0,49503.50,0
2,2,15694510,Hsueh,678,France,Male,40.0,10,0.00,2,1.0,0.0,184866.69,0
3,3,15741417,Kao,581,France,Male,34.0,2,148882.54,1,1.0,1.0,84560.88,0
4,4,15766172,Chiemenam,716,Spain,Male,33.0,5,0.00,2,1.0,1.0,15068.83,0


## 2. Removendo Colunas Desnecessárias

In [8]:
# Colunas que não agregam valor preditivo
cols_to_drop = ['id', 'CustomerId', 'Surname']
df = df.drop(columns=cols_to_drop)

print(f"Shape após remoção: {df.shape}")
print(f"Colunas restantes: {list(df.columns)}")

Shape após remoção: (165034, 11)
Colunas restantes: ['CreditScore', 'Geography', 'Gender', 'Age', 'Tenure', 'Balance', 'NumOfProducts', 'HasCrCard', 'IsActiveMember', 'EstimatedSalary', 'Exited']


## 3. Encoding de Variáveis Categóricas

In [9]:
# Geography: One-Hot Encoding (3 categorias sem ordem)
df = pd.get_dummies(df, columns=['Geography'], drop_first=False)

# Gender: Label Encoding (binário: Female=0, Male=1)
le = LabelEncoder()
df['Gender'] = le.fit_transform(df['Gender'])

print("Encoding concluído!")
print(f"Shape após encoding: {df.shape}")
print(f"\nColunas: {list(df.columns)}")

Encoding concluído!
Shape após encoding: (165034, 13)

Colunas: ['CreditScore', 'Gender', 'Age', 'Tenure', 'Balance', 'NumOfProducts', 'HasCrCard', 'IsActiveMember', 'EstimatedSalary', 'Exited', 'Geography_France', 'Geography_Germany', 'Geography_Spain']


## 4. Criando Novas Features

In [10]:
# Razão entre saldo e salário estimado
df['balance_salary_ratio'] = df['Balance'] / (df['EstimatedSalary'] + 1)

# Cliente com saldo zero
df['has_zero_balance'] = (df['Balance'] == 0).astype(int)

# Faixa etária
df['age_group'] = pd.cut(df['Age'], 
                          bins=[0, 30, 45, 60, 100], 
                          labels=['jovem', 'adulto', 'meia_idade', 'senior'])
df['age_group'] = LabelEncoder().fit_transform(df['age_group'])

# Cliente de alto risco (alemanha + mulher + meia idade)
df['high_risk'] = (
    (df['Geography_Germany'] == 1) & 
    (df['Gender'] == 0) & 
    (df['Age'].between(40, 60))
).astype(int)

print("Novas features criadas!")
print(f"Shape final: {df.shape}")
print(f"\nNovas colunas: ['balance_salary_ratio', 'has_zero_balance', 'age_group', 'high_risk']")

Novas features criadas!
Shape final: (165034, 17)

Novas colunas: ['balance_salary_ratio', 'has_zero_balance', 'age_group', 'high_risk']


## 5. Escalando Variáveis Numéricas

In [11]:
from sklearn.preprocessing import StandardScaler

# Colunas para escalar
cols_to_scale = ['CreditScore', 'Age', 'Tenure', 'Balance', 
                 'EstimatedSalary', 'balance_salary_ratio']

scaler = StandardScaler()
df[cols_to_scale] = scaler.fit_transform(df[cols_to_scale])

print("Escalonamento concluído!")
print(f"\nEstatísticas após escalonamento:")
print(df[cols_to_scale].describe().round(2))

Escalonamento concluído!

Estatísticas após escalonamento:
       CreditScore        Age     Tenure    Balance  EstimatedSalary  \
count    165034.00  165034.00  165034.00  165034.00        165034.00   
mean         -0.00       0.00      -0.00      -0.00            -0.00   
std           1.00       1.00       1.00       1.00             1.00   
min          -3.83      -2.27      -1.79      -0.88            -2.24   
25%          -0.74      -0.69      -0.72      -0.88            -0.75   
50%           0.03      -0.13      -0.01      -0.88             0.11   
75%           0.67       0.44       0.71       1.03             0.85   
max           2.42       6.08       1.77       3.11             1.74   

       balance_salary_ratio  
count             165034.00  
mean                   0.00  
std                    1.00  
min                   -0.02  
25%                   -0.02  
50%                   -0.02  
75%                   -0.01  
max                  140.12  


## 6. Salvando Dataset Processado

In [12]:
import joblib

# Salva o dataset processado
df.to_csv('../data/processed/train_processed.csv', index=False)

# Salva o scaler para usar na modelagem e no deploy
joblib.dump(scaler, '../models/scaler.pkl')

print("Dataset processado salvo em: data/processed/train_processed.csv")
print("Scaler salvo em: models/scaler.pkl")
print(f"\nShape final do dataset: {df.shape}")
print(f"\nColunas finais:\n{list(df.columns)}")

Dataset processado salvo em: data/processed/train_processed.csv
Scaler salvo em: models/scaler.pkl

Shape final do dataset: (165034, 17)

Colunas finais:
['CreditScore', 'Gender', 'Age', 'Tenure', 'Balance', 'NumOfProducts', 'HasCrCard', 'IsActiveMember', 'EstimatedSalary', 'Exited', 'Geography_France', 'Geography_Germany', 'Geography_Spain', 'balance_salary_ratio', 'has_zero_balance', 'age_group', 'high_risk']
